<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent06-top-light.svg" alt="A trace of one system in the batch draws along the MD steps, and five hooks fire in turn as it passes them: log, save frames, record a quantity, freeze atoms, and stop one system, where the trace ends." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 06</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Hooked in at every step</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Record your own quantities, hold atoms in place and stop a single system while a batch runs.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Made to order<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent06-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

A simulation is rarely just started and read at the end. You follow a
distance while it changes, hold the bottom of a slab in place, or stop a
trajectory that has gone wrong. In Toolkit each of these is a hook: a
small piece of code that acts at a chosen point of every step, on the GPU,
for every system in the batch.

In [ ]:
# Load the course helpers, three complexes, the composed model and the run settings
from helpers import agent06 as helpers
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent06_work
complexes, composed_model, fragment_a_sizes = helpers.prepare_complexes()
run_settings = dict(temperature=1000.0, dt=1.0, friction=0.01, n_steps=1000)  # K, fs, 1/fs, MD steps
helpers.show_session()                   # prints the session lines

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Watch a run</h2>

Three molecular complexes, each a pair of molecules held together by weak
interactions, run on the composed model of "Made to order" with
`run_settings`: a short run at 1000 K, hot enough to stress a weak pair.

Toolkit ships hooks for the common needs: a log of each system's energy
and temperature, saved frames, a check for broken numbers. Each one names
the point in the step where it acts and how often.

#### Try it: interaction energy during a run

Run the three complexes in one batch with `run_settings`. Built-in hooks
save the frames and write a log. Record the interaction energy of each
complex along the run, every 10 steps, in a unit of your choice (kcal/mol
is typical), and set it beside the value at the starting structure. In each
complex the first `fragment_a_sizes` atoms are fragment A; the rest are
fragment B.

| | |
| --- | --- |
| input  | `complexes`; `composed_model`; `run_settings`; `fragment_a_sizes` |
| output | `saved_step`, `interaction_energy`, `start_interaction_energy`, `plain_store`, `plain_log`, in `agent06_work/interaction.py` |

Done when: each panel shows the interaction energy along the run, with a
line for the starting value.

In [ ]:
%load agent06_work/interaction.py

In [ ]:
# Plot the interaction energy of each complex, then play the saved frames and show the log
helpers.plot_interaction(saved_step, interaction_energy, start_interaction_energy)
helpers.show_run(plain_store, plain_log)

Each panel follows one complex: its interaction energy against the MD step, with a dashed
line at the starting value. The viewer plays the saved frames. The table is the log: one row per
complex at each logged step, with its energy and temperature.

> </> API
>
> `engine.register_hook(hook)`
>
> **Input:** a hook: any object with a `stage`, a `frequency` and a call<br>
> **Effect:** Toolkit calls it at that stage, every `frequency` steps, with the live batch

> </> API
>
> `model.make_neighbor_hooks()`
>
> **Input:** none<br>
> **Effect:** returns the model's neighbor-list hooks; register them before `run()`

> </> API
>
> `DynamicsStage`
>
> **Input:** a hook's `stage`, for example `DynamicsStage.AFTER_STEP`<br>
> **Effect:** the point of the step where the hook fires. Model outputs change at `AFTER_COMPUTE`; observers and stoppers go at `AFTER_STEP`

> </> API
>
> `hook(ctx, stage)`
>
> **Input:** `ctx`, what a hook receives: `batch`, `step_count`, `model`, `active_graph_mask`, `workflow`<br>
> **Effect:** the hook reads and writes the live batch through `ctx.batch`

> ⚙️ **Fine control**
>
> ```python
> SnapshotHook(sink=..., frequency=...)                     # saved frames
> LoggingHook(backend="csv", log_path=..., frequency=...)   # one row per system
> ```
>
> Example: `save a frame every 5 steps`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. Record your own quantity</h2>

No built-in hook knows how far apart the two molecules of a complex are.
When your study needs a quantity of its own, your agent writes a hook for
it.

#### Try it: add separation and nearest contact

Repeat the plain run and record two distances for each complex, on the
steps of the interaction energy: the separation between the centers of mass of its two
fragments, and the nearest contact, the shortest distance between an atom
of one fragment and an atom of the other.

| | |
| --- | --- |
| input  | `complexes`; `composed_model`; `run_settings`; `fragment_a_sizes` |
| output | `separation`, `nearest_contact`, in `agent06_work/separation_hook.py` |

Done when: each panel shows both distances beside the interaction energy.

In [ ]:
%load agent06_work/separation_hook.py

In [ ]:
# Plot the interaction energy and both distances for each complex against the MD step
helpers.plot_separation(saved_step, interaction_energy, separation, nearest_contact)

Each column follows one complex: its interaction energy above, its fragment
separation and its nearest contact below, against the MD step.

> 💡 **Hint**
>
> Your agent reuses a built-in hook when one fits. Ask for something none
> of them records.

#### Try it: freeze atoms with a hook

In a surface simulation you usually fix the bottom layers of the slab so
they behave like the bulk crystal underneath. Chemists do the same with a
rigid framework that holds a guest molecule. Here you will fix fragment A
of each complex and let fragment B move around it.

Toolkit's `FreezeAtomsHook` does this: at every step it puts the chosen
atoms back where they started and clears their velocities and forces.

Your task. Ask your agent to repeat the plain run with fragment A of each
complex frozen.

| | |
| --- | --- |
| input  | `complexes`; `composed_model`; `run_settings`; `fragment_a_sizes` |
| output | `frozen_mask`, `frozen_store`, in `agent06_work/freeze.py` |

Done when: in the plot, every frozen atom stays at 0 Å from its start and
the free atoms move.

In [ ]:
%load agent06_work/freeze.py

In [ ]:
# Plot how far each atom moved from its start, plain run beside frozen run
helpers.plot_displacement(plain_store, frozen_store, frozen_mask)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. Stop one system</h2>

In a batch, one system can go wrong while the others are fine. A hook can
stop that one and let the rest run on.

#### Try it: retire a complex past a distance limit

Your plot in section 2 shows how far each nearest contact reaches. Read
off a limit that one complex passes and the others stay under. Repeat the
plain run and retire a complex once its nearest contact is above your
limit.

| | |
| --- | --- |
| input  | `complexes`; `composed_model`; `run_settings`; `fragment_a_sizes` |
| output | `stop_step`, `final_status`, `retire_store`, in `agent06_work/retire.py` |

Done when: the table gives a stop step for the complex that passed your
limit, and the others ran to the end.

> 💡 **Hint**
>
> The repeated run follows your plot only if it starts the same way. If no
> complex is retired, lower your limit.

In [ ]:
%load agent06_work/retire.py

In [ ]:
# Show which complex was retired, and when
helpers.show_retired(stop_step, final_status, retire_store)

The table gives the step at which each complex was retired, or shows that
it ran to the end. In the viewer a retired complex stands still while the
others move.

> </> API
>
> `batch.status`
>
> **Holds:** one number per system<br>
> **Effect:** a system whose status has reached the run's exit value is left untouched for the rest of the run. In a fused stage, the same hook goes on the fused stage.

**Bottom line:** A hook records, constrains or stops. A bias adds energy:
Toolkit runs it beside the model, with forces and stress from one energy function.

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

Official Toolkit examples that take hooks further:

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/05_safety_and_monitoring.py">Safety and timing hooks</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Catch broken numbers, clamp large forces, watch energy drift, time a run.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/02_custom_hook.py">A longer custom hook</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">A radial distribution function, recorded while the run goes.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/03_custom_convergence.py">Your own stopping rule</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>ConvergenceHook</code> with a criterion you write.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/12_umbrella_sampling.py">Walls and restraints</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">A wall or restraint is a bias, run by <code>EnhancedSampling</code>.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 06 recap</h2><div style="margin: 0 0 10px;">You attached built-in hooks to a run, had your agent write one for a quantity of your own, froze part of each complex, and stopped one system while the others ran on.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-07-one-out-one-in.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">One out, one in →</a></div></div>